In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv mysql-connector-python requests

import os
import base64
import uvicorn
import nest_asyncio
from fastapi import FastAPI, Request, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from openai import OpenAI
import ollama
import config
import database

# Jupyter 환경에서 FastAPI 실행을 위한 설정
nest_asyncio.apply()

app = FastAPI()

# CORS 설정: 모든 Origin/Method/Header 허용 (*)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def saveToDatabase(userQuestion, aiResponse, imagePath):
    """ 분석 결과를 MySQL 데이터베이스에 영구 저장합니다. """
    try:
        query = "INSERT INTO analysis_history (question, response, imagePath) VALUES (%s, %s, %s)"
        params = (userQuestion, aiResponse, imagePath)
        saveResult = database.executeQuery(query, params)
        return saveResult
    except Exception as e:
        print(f"Database saving failed: {e}")
        return {"success": False, "message": str(e)}

async def analyzeWithGpt(imageBytes, userQuestion):
    """ OpenAI GPT-4o 모델을 사용하여 이미지 분석을 수행합니다. """
    appConfig = config.getAppConfig()
    client = OpenAI(api_key=appConfig["openaiApiKey"])
    
    # 이미지를 Base64 문자열로 인코딩
    base64Image = base64.b64encode(imageBytes).decode('utf-8')
    
    chatResponse = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": userQuestion},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64Image}"}
                    }
                ]
            }
        ]
    )
    return chatResponse.choices[0].message.content

async def analyzeWithOllama(imageBytes, userQuestion):
    """ 로컬 Ollama 모델(gemma4:e2b)을 사용하여 이미지 분석을 수행합니다. """
    appConfig = config.getAppConfig()
    modelName = appConfig["ollamaModel"]
    
    ollamaResponse = ollama.chat(
        model=modelName,
        messages=[{
            'role': 'user',
            'content': userQuestion,
            'images': [imageBytes]
        }]
    )
    return ollamaResponse['message']['content']

@app.post("/analyze")
async def analyzeImage(file: UploadFile = File(...), question: str = Form(...)):
    """ 사용자가 업로드한 이미지와 질문을 처리하여 분석 결과를 반환합니다. """
    try:
        # 환경 설정 로드
        appConfig = config.getAppConfig()
        useModel = appConfig["useModel"]
        
        # 파일 저장 처리
        savePath = os.path.join("dataset", file.filename)
        fileContent = await file.read()
        
        with open(savePath, "wb") as imageFile:
            imageFile.write(fileContent)
            
        # 모델 선택 및 분석 실행
        finalResult = ""
        if useModel == "GPT":
            finalResult = await analyzeWithGpt(fileContent, question)
        elif useModel == "OLLAMA":
            finalResult = await analyzeWithOllama(fileContent, question)
        else:
            raise Exception("유효하지 않은 모델 설정입니다. (.env의 USE_MODEL 확인)")
            
        # 분석 결과 저장
        #saveToDatabase(question, finalResult, savePath)
        
        return {"success": True, "data": finalResult}
        
    except Exception as e:
        # 가이드라인에 따른 규격화된 에러 JSON 반환
        return JSONResponse(
            status_code=500,
            content={"success": False, "message": str(e)}
        )

import nest_asyncio
nest_asyncio.apply()

uvi_config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(uvi_config)
await server.serve()

INFO:     Started server process [29224]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:56083 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:56083 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:65415 - "POST /analyze HTTP/1.1" 500 Internal Server Error
